In [ ]:
import numpy as np
import time
import torch

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

# ==========================================
# 1. ТРАДИЦИОННЫЙ VM-ПОДХОД (Фон Нейман)
# ==========================================
def run_von_neumann_vm(n_steps=1_000_000):
    # Память данных и память инструкций
    data_mem = np.zeros(256, dtype=np.float32)
    # Опкоды: 0: ADD, 1: MUL, 2: XOR-approx, 3: LOAD/STORE
    instructions = np.random.randint(0, 4, size=(n_steps, 3), dtype=np.int32)

    start_time = time.perf_counter()
    pc = 0
    # Простой интерпретатор со счетчиком команд
    while pc < n_steps:
        op, r1, r2 = instructions[pc]
        if op == 0:
            data_mem[r1 % 256] += data_mem[r2 % 256] + 0.1
        elif op == 1:
            data_mem[r1 % 256] = (data_mem[r1 % 256] * 0.99) + 0.01
        elif op == 2:
            data_mem[r1 % 256] = np.sin(data_mem[r2 % 256])
        elif op == 3:
            data_mem[r1 % 256] = data_mem[r2 % 256]
        pc += 1
    elapsed = time.perf_counter() - start_time
    return elapsed, data_mem

# ==========================================
# 2. 3D SPATIAL DATA ENGINE (Тензорный бульон)
# ==========================================
def run_spatial_3d_engine(cube_size=64, n_steps=1000):
    # Инициализация 3D-пространства данных: (C, D, H, W)
    # Каналы: 0 - Значение, 1 - Аффинность (маска), 2 - Градиент сдвига
    space = torch.rand((1, 3, cube_size, cube_size, cube_size), device=device, dtype=torch.float32)

    # 3D Ядро взаимодействия (аналог считывающего фермента / локального правила)
    kernel = torch.randn((3, 3, 3, 3, 3), device=device, dtype=torch.float32) * 0.1

    start_time = time.perf_counter()
    if device.type == "cuda":
        torch.cuda.synchronize()

    for _ in range(n_steps):
        # Локальная 3D-свертка: данные трансформируют сами себя на основе соседей
        # без единой классической инструкции перехода
        interaction = torch.nn.functional.conv3d(space, kernel, padding=1)
        space = torch.tanh(space + 0.05 * interaction)

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    total_voxel_ops = n_steps * (cube_size ** 3) * 27 * 3  # Свертка 3x3x3 по объему
    return elapsed, space, total_voxel_ops

# ==========================================
# ЗАПУСК И СРАВНЕНИЕ
# ==========================================
print("\n--- Запуск традиционной VM (CPU Fetch-Decode Loop) ---")
vn_time, vn_mem = run_von_neumann_vm(n_steps=2_000_000)
vn_mops = (2_000_000 / vn_time) / 1e6
print(f"Время: {vn_time:.4f} сек | Скорость: {vn_mops:.2f} Mops/sec")

print("\n--- Запуск 3D Spatial Dataflow Engine (Tensor/GPU) ---")
sp_time, sp_space, sp_ops = run_spatial_3d_engine(cube_size=64, n_steps=1000)
sp_gops = (sp_ops / sp_time) / 1e9
print(f"Время: {sp_time:.4f} сек | Эффективная производительность: {sp_gops:.2f} GOPS (млрд операций/сек)")

PyTorch version: 2.11.0+cpu
Active Device: cpu

--- Запуск традиционной VM (CPU Fetch-Decode Loop) ---
Время: 6.7887 сек | Скорость: 0.29 Mops/sec

--- Запуск 3D Spatial Dataflow Engine (Tensor/GPU) ---
Время: 63.5937 сек | Эффективная производительность: 0.33 GOPS (млрд операций/сек)
